In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [33]:
df=pd.read_csv("/content/train.csv")
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [34]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [35]:
#df['Cabin'].value_counts()

In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:

df['Age']=df['Age'].fillna(df['Age'].median())
df['Cabin']=df['Cabin'].fillna(df['Cabin'].mode()[0])
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])

In [38]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [39]:
df['Cabin'].sample(5)

,Cabin
806,A36
208,B96 B98
829,B28
156,B96 B98
786,B96 B98


In [40]:

df.duplicated().sum()

np.int64(0)

In [41]:
cat_data=df.select_dtypes(include='object')
num_data=df.select_dtypes(include='number')

In [42]:

cat_data.head()

,Name,Sex,Ticket,Cabin,Embarked
0,"Braund, Mr. Owen Harris",male,A/5 21171,B96 B98,S
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,PC 17599,C85,C
2,"Heikkinen, Miss. Laina",female,STON/O2. 3101282,B96 B98,S
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,113803,C123,S
4,"Allen, Mr. William Henry",male,373450,B96 B98,S


In [43]:
cat_data.drop(columns=['Name','Ticket','Cabin'],inplace=True)

In [44]:
cat_data.head()

,Sex,Embarked
0,male,S
1,female,C
2,female,S
3,female,S
4,male,S


In [45]:
encoded_cat_data=pd.get_dummies(cat_data,drop_first=True,dtype=int)

In [46]:
encoded_cat_data.head()

,Sex_male,Embarked_Q,Embarked_S
0,1,0,1
1,0,0,0
2,0,0,1
3,0,0,1
4,1,0,1


In [47]:
final_df=pd.concat([num_data,encoded_cat_data],axis=1)

In [48]:
final_df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,22.0,1,0,7.2500,1,0,1
1,2,1,1,38.0,1,0,71.2833,0,0,0
2,3,1,3,26.0,0,0,7.9250,0,0,1
3,4,1,1,35.0,1,0,53.1000,0,0,1
4,5,0,3,35.0,0,0,8.0500,1,0,1


In [49]:
from sklearn.model_selection import train_test_split
x=final_df.drop(columns=['Survived'])
y=final_df['Survived']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=100)

In [50]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
rf_pred=rf.predict(x_test)

In [51]:
from sklearn.metrics import classification_report
print(classification_report(y_test,rf_pred))

              precision    recall  f1-score   support

           0       0.83      0.90      0.87       104
           1       0.85      0.75      0.79        75

    accuracy                           0.84       179
   macro avg       0.84      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



In [52]:
test_df=pd.read_csv("/content/test.csv")
test_df.head(2)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S


In [53]:
test_df.isnull().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,86
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,327


In [54]:
test_df['Age']=test_df['Age'].fillna(test_df['Age'].median())
test_df['Fare']=test_df['Fare'].fillna(test_df['Fare'].median())

In [55]:
testcat_data=test_df.select_dtypes(include='object')
testnum_data=test_df.select_dtypes(include='number')

In [56]:
testcat_data.drop(columns=['Name','Ticket','Cabin'],inplace=True)

In [57]:
encoded_testcat_data=pd.get_dummies(testcat_data,drop_first=True,dtype=int)

In [58]:
final_test_df=pd.concat([testnum_data,encoded_testcat_data],axis=1)

In [59]:
final_test_df.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,892,3,34.5,0,0,7.8292,1,1,0
1,893,3,47.0,1,0,7.0000,0,0,1
2,894,2,62.0,0,0,9.6875,1,1,0


In [60]:
predictions=rf.predict(final_test_df)

In [66]:
predictions_df=pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':predictions})

In [67]:
len(predictions_df)

418

In [68]:
predictions_df.to_csv('RFsubmission.csv',index=False)
print('Submission file created successfully!')

Submission file created successfully!


In [69]:
#hypertunedRandomforestclassifier

In [70]:
from sklearn.model_selection import GridSearchCV
parameters={'n_estimators':[100,200,300,500],'max_depth':[2,3,4],'criterion':['gini','entropy'],'max_features':['sqrt','log2'],'max_leaf_nodes':[2,3,4]}
rf_clf=RandomForestClassifier()
rf_grid=GridSearchCV(rf_clf,parameters,cv=3,n_jobs=-1,scoring='f1_macro')
rf_grid.fit(x,y)


GridSearchCV(cv=3, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 3, 4],
                         'max_features': ['sqrt', 'log2'],
                         'max_leaf_nodes': [2, 3, 4],
                         'n_estimators': [100, 200, 300, 500]},
             scoring='f1_macro')

In [71]:
rf_grid.best_score_

np.float64(0.7678029243028659)

In [73]:
rf_grid.best_params_

{'criterion': 'entropy',
 'max_depth': 3,
 'max_features': 'sqrt',
 'max_leaf_nodes': 4,
 'n_estimators': 100}

In [74]:
hyrf_clf=RandomForestClassifier(criterion='entropy',max_depth=3,max_features='sqrt',max_leaf_nodes=4,n_estimators=100)
hyrf_clf.fit(x_train,y_train)

RandomForestClassifier(criterion='entropy', max_depth=3, max_leaf_nodes=4)

In [75]:
hyrf_clf_pred=hyrf_clf.predict(x_test)

In [77]:
print(classification_report(y_test,hyrf_clf_pred))

              precision    recall  f1-score   support

           0       0.73      0.93      0.82       104
           1       0.85      0.53      0.66        75

    accuracy                           0.77       179
   macro avg       0.79      0.73      0.74       179
weighted avg       0.78      0.77      0.75       179



In [78]:
hyrf_clf_final_pred=hyrf_clf.predict(final_test_df)

In [79]:
final_pred=pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':hyrf_clf_final_pred})

In [80]:
final_pred.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [81]:
final_pred.to_csv('hypertunedRFsubmission.csv',index=False)
print('Submission file created successfully!')

Submission file created successfully!
